# Week 3

THe goal of this week is to: 

1. [] Confirm that me and Jan gets the same likelihood values given the same parameters
2. [] Research why the phi values in notebook week_2 is over 1 and what this means
3. [] Create a profile likelihood plot of phi 
4. [] Make mu non-free in the sense that thay are bounded by a sorted order
5. [] Interpretet the states and make sure they make sense

## Config

In [1]:
import jax 
from src.data import load_model_and_data 
from src.optim.loss import negative_log_likelihood

jax.config.update("jax_enable_x64", True)  # Enable 64-bit precision for JAX 

class MODELTYPES: 
    StationaryHMM = "stationary_hmm_4_state" 
    AR1 = "ar1_model" 
    AR2 = "ar2_model" 
    AR_HMM = "ar_hmm_model"
    COVARIATE_HMM = "covariate_hmm_model"
    COVARIATE_AR_HMM = "covariate_ar_hmm_model"  



RUN_TO_LOAD = {
    "tag": "week_2",
    "run": 1
}


## 1. Confirming Likelihoods and model params

Checked that me and Jan has the exact same y values

### Stationary HMM

In [2]:
model, y, X = load_model_and_data(modelname=MODELTYPES.StationaryHMM, tag=RUN_TO_LOAD["tag"], run=RUN_TO_LOAD["run"])
model 

StationaryHMM(transition: StationaryTransition(
  transition_logits=f64[4,3], initial_state_dist=None, num_states=4
), emission: GaussianEmisionBackground(mu=f64[3], log_sigma=f64[4]))

In [3]:
mu, sigma, transition_matrix = model.mu(), model.sigma(), model.transition_matrix()

for i, (m, s) in enumerate(zip(mu, sigma)):
    print(f"State {i}: mu={m:.4f}, sigma={s:.4f}") 

State 0: mu=400.0000, sigma=21.7280
State 1: mu=535.6474, sigma=76.5895
State 2: mu=890.0040, sigma=111.0071
State 3: mu=1306.5267, sigma=197.3977


![Results for 4 states](images/jans_book/stationary/results_4_states.png)

In [4]:
print("Transition Matrix:")
print(transition_matrix)

print("State distribution")
print(model.transition.u0())

Transition Matrix:
[[8.49417101e-01 1.47756510e-01 2.82638881e-03 2.14897417e-17]
 [8.25318826e-02 8.49531488e-01 6.79366297e-02 6.60630290e-15]
 [4.27781253e-04 1.61716562e-01 6.84203524e-01 1.53652133e-01]
 [3.14348210e-13 4.29480427e-16 7.23388998e-02 9.27661100e-01]]
State distribution
[0.19019046 0.34624139 0.14838646 0.31518169]


![Results for 4 states](images/jans_book/stationary/results_transition.png)

In [5]:
JANS_LIKELIHOOD = -10389.0 

print(f"Log-Likelihood: {-negative_log_likelihood(model, y, X):.4f}")

Log-Likelihood: -10388.6807


### Auto Regressive Model

In [6]:
model, y, X = load_model_and_data(modelname=MODELTYPES.AR2, tag=RUN_TO_LOAD["tag"], run=RUN_TO_LOAD["run"])

JANS_LIKELIHOOD = -10207.0 

print(f"Log-Likelihood of AR(2) model: {model.llf:.4f}") 
print(f"Jans log-likelihood: {JANS_LIKELIHOOD:.4f}")
print(model.summary())  



Log-Likelihood of AR(2) model: -10207.1111
Jans log-likelihood: -10207.0000
                               SARIMAX Results                                
Dep. Variable:                      y   No. Observations:                 1666
Model:                 ARIMA(2, 0, 0)   Log Likelihood              -10207.111
Date:                Fri, 06 Mar 2026   AIC                          20422.222
Time:                        11:53:05   BIC                          20443.895
Sample:                             0   HQIC                         20430.254
                               - 1666                                         
Covariance Type:                  opg                                         
                 coef    std err          z      P>|z|      [0.025      0.975]
------------------------------------------------------------------------------
const        805.6580     48.949     16.459      0.000     709.719     901.597
ar.L1          1.3033      0.015     84.654      0.000 

|        | ar1    | ar2     | intercept |
|--------|--------|---------|-----------|
| value  | 1.3033 | -0.3696 | 805.7683  |
| s.e.   | 0.0228 | 0.0228  | 40.7347   |

We get the exact same vals

### HMM Autoregressive model

In [7]:
import jax.numpy as jnp 

model, y, X = load_model_and_data(modelname=MODELTYPES.AR_HMM, tag=RUN_TO_LOAD["tag"], run=RUN_TO_LOAD["run"]) 

model

ArHMM(transition: StationaryTransition(
  transition_logits=f64[4,3], initial_state_dist=None, num_states=4
), emission: ArGaussianEmisionBackground(mu=f64[3], log_sigma=f64[4], phi=f64[4]))

In [8]:
JANS_LIKELIHOOD = -9354.0 
print(f"Log-Likelihood of AR-HMM model: {-negative_log_likelihood(model, y, X):.4f}")
print(f"Jans log-likelihood: {JANS_LIKELIHOOD:.4f}")

Log-Likelihood of AR-HMM model: -9145.6768
Jans log-likelihood: -9354.0000


In [9]:
mu, sigma, transition_matrix = model.emission.mu, jnp.exp(model.emission.log_sigma), model.transition.transition_matrix
phi = model.emission.phi 
mu = jnp.concatenate([jnp.array([400.0]), mu])  # Add zero for the first state (AR(2) state)

for i, (m, s) in enumerate(zip(mu, sigma)):
    print(f"State {i}: mu={m:.4f}, sigma={s:.4f}, phi={phi[i]:.4f}")

State 0: mu=400.0000, sigma=0.4363, phi=1.0001
State 1: mu=535.6805, sigma=39.5384, phi=1.0379
State 2: mu=889.9311, sigma=187.6028, phi=0.7414
State 3: mu=1306.7014, sigma=39.9664, phi=0.6426


![Results for 4 states ArrHmm](images/jans_book/arr_hmm/params.png)

We now test that given Jans parameters, we will get the same likelihood

In [10]:
from src.models.ar_hmm import ArHMM  

#JANS RESULTS 
mu = jnp.array([453, 730, 1590]) 
sigma = jnp.array([13, 42, 108, 72]) 
phi = jnp.array([0.97, 0.30, 0.62, 0.85 ]) 


Gamma = jnp.array([
    [8.2e-01, 0.0946, 0.0870, 9.3e-09],
    [1.8e-01, 0.6875, 0.1360, 5.5e-09],
    [9.6e-02, 0.2104, 0.4140, 2.8e-01],
    [2.3e-08, 0.0049, 0.1330, 8.6e-01],
])

stat_dist = jnp.array([[0.29, 0.21, 0.17, 0.34]])


## CONVERTING TO MODEL PARAMS 
num_states = 4
tGamma = jnp.zeros((num_states, num_states))
tGamma = tGamma.at[jnp.diag_indices(num_states)].set(0.0)
rows, cols = jnp.where(~jnp.eye(num_states, dtype=bool), size=num_states * (num_states - 1))
tGamma = tGamma.at[rows, cols].set(jnp.log(Gamma[rows, cols])) 

offdiag_mask = ~jnp.eye(num_states, dtype=bool)

# From Gamma directly (4x4 -> 4x3)
Gamma_no_diag = Gamma[offdiag_mask].reshape(num_states, num_states - 1)
# If you want it from tGamma instead:
tGamma_no_diag = tGamma[offdiag_mask].reshape(num_states, num_states - 1)



ar_hmm_jans = ArHMM(mu=mu, log_sigma=jnp.log(sigma), phi=phi, transition_logits=tGamma_no_diag)


for i, (m_jan, m, s_jan, s, p_jan, p) in enumerate(zip(mu, ar_hmm_jans.emission.mu, sigma, ar_hmm_jans.emission.log_sigma, phi, ar_hmm_jans.emission.phi)):
    
    print(f"State {i}:")
    print(f"  Jans mu: {m_jan:.4f}, Model mu: {m:.4f}")
    print(f"  Jans sigma: {s_jan:.4f}, Model sigma: {jnp.exp(s):.4f}")
    print(f"  Jans phi: {p_jan:.4f}, Model phi: {p:.4f}")
    print()


Gamma_from_model = ar_hmm_jans.transition.transition_matrix()

for i in range(num_states):
    print(f"State {i} transition probabilities:") 
    for j in range(num_states):
        print(f"  To state {j}: {Gamma_from_model[i, j]:.4f} (Jans: {Gamma[i, j]:.4f})")


print("Stationary distribution from model:")
print(ar_hmm_jans.transition.u0())
print("Stationary distribution from Jans:")
print(stat_dist)

State 0:
  Jans mu: 453.0000, Model mu: 453.0000
  Jans sigma: 13.0000, Model sigma: 13.0000
  Jans phi: 0.9700, Model phi: 0.9700

State 1:
  Jans mu: 730.0000, Model mu: 730.0000
  Jans sigma: 42.0000, Model sigma: 42.0000
  Jans phi: 0.3000, Model phi: 0.3000

State 2:
  Jans mu: 1590.0000, Model mu: 1590.0000
  Jans sigma: 108.0000, Model sigma: 108.0000
  Jans phi: 0.6200, Model phi: 0.6200

State 0 transition probabilities:
  To state 0: 0.8463 (Jans: 0.8200)
  To state 1: 0.0801 (Jans: 0.0946)
  To state 2: 0.0736 (Jans: 0.0870)
  To state 3: 0.0000 (Jans: 0.0000)
State 1 transition probabilities:
  To state 0: 0.1368 (Jans: 0.1800)
  To state 1: 0.7599 (Jans: 0.6875)
  To state 2: 0.1033 (Jans: 0.1360)
  To state 3: 0.0000 (Jans: 0.0000)
State 2 transition probabilities:
  To state 0: 0.0605 (Jans: 0.0960)
  To state 1: 0.1326 (Jans: 0.2104)
  To state 2: 0.6304 (Jans: 0.4140)
  To state 3: 0.1765 (Jans: 0.2800)
State 3 transition probabilities:
  To state 0: 0.0000 (Jans: 0.00

In [11]:
import equinox as eqx 
from src.optim.minimizer import Minimizer 


class ArHmmProfile(ArHMM): 
    def __init__(self, mu, log_sigma, phi, transition_logits):
        super().__init__(mu=mu, log_sigma=log_sigma, phi=phi, transition_logits=transition_logits) 

    def filter_spec(self):
        spec = jax.tree_util.tree_map(eqx.is_inexact_array, self)

        # freeze initial state dist
        spec = eqx.tree_at(
            lambda m: m.transition.initial_state_dist,
            spec,
            replace=False,
        )

        # freeze only phi[0], keep phi[1:] trainable
        phi_mask = jnp.ones_like(self.emission.phi, dtype=bool).at[0].set(False)
        spec = eqx.tree_at(
            lambda m: m.emission.phi,
            spec,
            replace=phi_mask,
        )
        return spec 
    

def profile(model: ArHMM, y, X, phi_range: tuple, num_points: int = 10):
    phi_space = jnp.linspace(phi_range[0], phi_range[1], num=num_points)
    log_like = jnp.zeros(num_points)

    for i in range(num_points):
        phi0 = phi_space[i]

        # fresh model each grid point
        m0 = ArHmmProfile(
            mu=model.emission.mu,
            log_sigma=model.emission.log_sigma,
            phi=model.emission.phi,
            transition_logits=model.transition.transition_logits,
        )

        def loss_fixed_phi0(m, y, X):
            phi_fixed = m.emission.phi.at[0].set(phi0)
            m_fixed = eqx.tree_at(lambda z: z.emission.phi, m, phi_fixed)
            return negative_log_likelihood(m_fixed, y, X)

        optimizer = Minimizer(loss_fn=loss_fixed_phi0, model=m0)
        opt_model = optimizer.run(y, X)

        log_like = log_like.at[i].set(-loss_fixed_phi0(opt_model, y, X))

    return phi_space, log_like


    


phi_range = (-2, 2) 
phi_space, log_like = profile(ar_hmm_jans, y, X, phi_range=phi_range, num_points=20)

ValueError: `filter_spec` must consist of booleans and callables only.